In [1]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
from scipy import stats
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from ols_diagnostics import full_diagnostic_report, validation_report

In [2]:
df = pd.read_csv('pullman_data.csv')
df.head()

,Date,DayName,OwnPrice (Rp),CompetitorPrice (Rp),Occupancy (%)
0,2025-01-01,Wednesday,1499179,1490434,99.4
1,2025-01-02,Thursday,1755757,1466462,73.0
2,2025-01-03,Friday,1769803,1716859,65.9
3,2025-01-04,Saturday,1812768,1660975,95.3
4,2025-01-05,Sunday,1722589,1226016,82.8


In [3]:
# 1. Load & Parsing Tanggal
df['Date'] = pd.to_datetime(df['Date'])

# 2. Extract Calendar Features
df['Month'] = df['Date'].dt.month
df['DayOfWeek'] = df['Date'].dt.dayofweek
df["IsWeekend"] = df["DayOfWeek"].isin([5, 6]).astype(int)
df['PriceRatio'] = df['OwnPrice (Rp)'] / df['CompetitorPrice (Rp)'].replace(0, np.nan)
df['PriceDiff'] = df['OwnPrice (Rp)'] - df['CompetitorPrice (Rp)']
df['IsFriday']   = (df['DayOfWeek'] == 4).astype(int)
df['OwnPrice_centered'] = df['OwnPrice (Rp)'] - df['OwnPrice (Rp)'].mean()
df['Price_Weekend'] = df['OwnPrice_centered'] * df['IsWeekend']

# 1. Pastikan tidak ada nilai 0 atau negatif sebelum dilog-kan
df['OwnPrice_clean'] = df['OwnPrice (Rp)'].replace(0, np.nan)
df['CompetitorPrice_clean'] = df['CompetitorPrice (Rp)'].replace(0, np.nan)

# 2. Buat variabel Log-Log (X1 dan X2)
df['ln_OwnPrice'] = np.log(df['OwnPrice_clean'])
df['ln_CompetitorPrice'] = np.log(df['CompetitorPrice_clean'])

m = df["Date"].dt.month
d = df["Date"].dt.day

df["IsHighSeason"] = (
    ((m == 1) & (d <= 7))
    | ((m == 6) | (m == 7))
    | ((m == 12) & (d >= 16))
).astype(int)

# 5. Tentukan Feature Set (Hindari Multikolinearitas)
feature_cols = [
    'OwnPrice (Rp)',
    # 'ln_OwnPrice',
    # 'ln_CompetitorPrice',
    # 'CompetitorPrice (Rp)',
    # 'PriceDiff',
    'PriceRatio',
    'IsWeekend',
    # 'IsFriday',
    # 'Price_Weekend',
    'IsHighSeason'
]

X = df[feature_cols]
y = df['Occupancy (%)']

# Train-Test Split (273 data pertama = Jan - Sep 2025)
X_train = X.iloc[0:273]
X_test  = X.iloc[273:]
y_train = y.iloc[0:273]
y_test  = y.iloc[273:]

In [4]:
# X_train sudah dalam bentuk DataFrame numerik (tanpa konstanta, itu ditambah otomatis)
model, results = full_diagnostic_report(
    X_train, y_train,
    feature_names=feature_cols,
    y_name="Occupancy",
    model_label="Model 3 Ratio (OwnPrice, PriceRatio, IsWeekend, IsHighSeason)"
)

# Validasi ke data Okt-Des
val = validation_report(model, X_test, y_test, label="Okt-Des 2025")


######################################################################
#  FULL DIAGNOSTIC REPORT - Model 3 Ratio (OwnPrice, PriceRatio, IsWeekend, IsHighSeason)
######################################################################

1. PEARSON CORRELATION MATRIX (antar variabel independen)
               OwnPrice (Rp)  PriceRatio  IsWeekend  IsHighSeason
OwnPrice (Rp)          1.000       0.553      0.329         0.321
PriceRatio             0.553       1.000      0.092        -0.012
IsWeekend              0.329       0.092      1.000        -0.008
IsHighSeason           0.321      -0.012     -0.008         1.000

--- Korelasi tiap X terhadap Y (Occupancy) ---
     Variabel       r  p-value Sig
OwnPrice (Rp)  0.2582   0.0000 ***
   PriceRatio -0.1905   0.0016  **
    IsWeekend  0.5741   0.0000 ***
 IsHighSeason  0.3818   0.0000 ***

2. MODEL SUMMARY (Coefficients, t-test, F-test)
                  Coef  Std Error   t-stat  p-value  Sig
const          67.9682     5.6114  12.1125   0.00

c:\Users\ASUS\Downloads\piton\statistik\final_model\ols_diagnostics.py:186: FutureWarning: acorr_breusch_godfrey currently returns a plain tuple whose length depends on the store argument. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an LMTestResult NamedTuple. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  bg_stat, bg_p, bg_f, bg_fp = acorr_breusch_godfrey(model, nlags=nlags)


In [5]:
# 1. Cek observasi 239 (Cook's D = 0.246, jauh lebih tinggi dari model2 lain)
print(df.iloc[239])

# 2. HAC untuk Model 2
X_train_const_m2 = sm.add_constant(X_train)  # OwnPrice, CompPrice, IsWeekend, IsHighSeason
model_m2_hac = sm.OLS(y_train, X_train_const_m2).fit(cov_type='HAC', cov_kwds={'maxlags': 7})
print(model_m2_hac.summary())

Date                     2025-08-28 00:00:00
DayName                             Thursday
OwnPrice (Rp)                        2641321
CompetitorPrice (Rp)                 2395691
Occupancy (%)                          100.0
Month                                      8
DayOfWeek                                  3
IsWeekend                                  0
PriceRatio                           1.10253
PriceDiff                             245630
IsFriday                                   0
OwnPrice_centered             1379996.838356
Price_Weekend                            0.0
OwnPrice_clean                       2641321
CompetitorPrice_clean                2395691
ln_OwnPrice                         14.78679
ln_CompetitorPrice                 14.689182
IsHighSeason                               0
Name: 239, dtype: object
                            OLS Regression Results                            
Dep. Variable:          Occupancy (%)   R-squared:                       0.547
Model: 

In [6]:
import contextlib

with open("model3ratio_report.txt", "w") as f:
    with contextlib.redirect_stdout(f):
        model, results = full_diagnostic_report(
            X_train, y_train,
            feature_names=feature_cols,
            y_name="Occupancy",
            model_label="Model 3 Ratio (OwnPrice, PriceRatio, IsWeekend, IsHighSeason)"
        )
        val = validation_report(model, X_test, y_test, label="Okt-Des 2025")

print("Report tersimpan")

Report tersimpan


c:\Users\ASUS\Downloads\piton\statistik\final_model\ols_diagnostics.py:186: FutureWarning: acorr_breusch_godfrey currently returns a plain tuple whose length depends on the store argument. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an LMTestResult NamedTuple. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  bg_stat, bg_p, bg_f, bg_fp = acorr_breusch_godfrey(model, nlags=nlags)
